# ANN Salary Regression — Hyperparameter Tuning

This notebook explores whether changing the neural network architecture can improve the salary-regression experiment.

The base model uses a simple ANN with two hidden layers. Here, instead of choosing the architecture manually, I will systematically test different numbers of hidden layers and neurons using `GridSearchCV` and SciKeras.

## What hyperparameter tuning means

A **hyperparameter** is a setting chosen before training rather than learned directly from the data.

In this experiment, I will tune:

- Number of hidden layers
- Number of neurons per hidden layer
- Number of training epochs

The purpose is not to make the model look more complex. The purpose is to check whether a different ANN configuration can generalize better.

The final test set will remain untouched during the search and will only be used after the best configuration has been selected.

## 1. Import Libraries

I am using:

- Pandas for tabular data handling
- Scikit-learn for splitting, preprocessing, cross-validation, and evaluation
- SciKeras to connect Keras models with Scikit-learn's `GridSearchCV`
- TensorFlow/Keras to build the ANN
- NumPy for numerical calculations

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from scikeras.wrappers import KerasRegressor

import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

import pickle

## 2. Load and Prepare the Dataset

I am using the same `Churn_Modelling.csv` dataset used in the main salary regression notebook.

The target for this experiment is `EstimatedSalary`.

The `Exited` column is deliberately excluded because it belongs to the separate churn-classification problem.

In [3]:
# Load the raw bank-customer dataset.
data = pd.read_csv(
    "/workspaces/Ann-churn-prediction/data/Churn_Modelling.csv"
)

# Remove record identifiers that are not useful model features.
data = data.drop(
    ["RowNumber", "CustomerId", "Surname"],
    axis=1
)

# Encode Gender using the same approach as the main regression notebook.
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(
    data["Gender"]
)

# One-hot encode Geography.
one_hot_encoder_geo = OneHotEncoder(
    handle_unknown="ignore"
)

geo_encoded = one_hot_encoder_geo.fit_transform(
    data[["Geography"]]
).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=one_hot_encoder_geo.get_feature_names_out(
        ["Geography"]
    )
)

# Replace the original Geography column with the encoded columns.
data = pd.concat(
    [
        data.drop("Geography", axis=1),
        geo_encoded_df
    ],
    axis=1
)

# Define features and regression target.
X = data.drop(
    ["EstimatedSalary", "Exited"],
    axis=1
)

y = data["EstimatedSalary"]

print("Feature shape:", X.shape)
print("Target shape :", y.shape)

Feature shape: (10000, 11)
Target shape : (10000,)


## 3. Create Training and Test Sets

The final test set must remain untouched while hyperparameters are being selected.

I will therefore split the data once:

- **Training set** → used by cross-validation during hyperparameter search
- **Test set** → reserved for the final evaluation

The scaler will live inside the Scikit-learn pipeline, so each CV fold fits its own scaler using only that fold's training portion.

In [4]:
# Keep a final hold-out test set for the end of the experiment.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set    :", X_test.shape)

Training set: (8000, 11)
Test set    : (2000, 11)


## 4. Define the ANN Builder

`GridSearchCV` needs a function that can build a fresh neural network for every hyperparameter combination.

The function below allows the search process to vary the number of neurons and hidden layers.

The output layer contains one linear neuron because `EstimatedSalary` is a continuous regression target.

In [5]:
def build_regression_model(neurons=32, layers=2):
    # Start a fresh Sequential model for every search candidate.
    model = Sequential()

    # Explicit Input layer avoids Keras input-shape warnings.
    model.add(Input(shape=(X_train.shape[1],)))

    # Create the requested number of hidden layers.
    for _ in range(layers):
        model.add(
            Dense(
                neurons,
                activation="relu"
            )
        )

    # One linear output for continuous salary prediction.
    model.add(Dense(1))

    # Compile for regression.
    model.compile(
        optimizer="adam",
        loss="mean_absolute_error",
        metrics=["mae"]
    )

    return model

## 5. Wrap the ANN in a Scikit-learn Pipeline

The pipeline contains two stages:

1. `StandardScaler` scales the features.
2. `KerasRegressor` trains the ANN.

Keeping the scaler inside the pipeline is important because `GridSearchCV` performs cross-validation on the training set. Each fold should learn scaling parameters only from its own training portion.

In [6]:
# Create a Scikit-learn pipeline.
pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ann",
        KerasRegressor(
            model=build_regression_model,
            verbose=0,
            random_state=42
        )
    )
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('ann', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,model,<function bui...x7d70a12a6200>
,random_state,42
,verbose,0
,build_fn,None


## 6. Define the Hyperparameter Grid

Instead of manually guessing one architecture, I will test a small, interpretable search space.

The search will compare:

- 16, 32, 64 neurons
- 1 or 2 hidden layers
- 50 or 100 epochs

This creates a finite set of experiments that can be compared using the same cross-validation procedure.

In [7]:
param_grid = {
    "ann__model__neurons": [16, 32, 64],
    "ann__model__layers": [1, 2],
    "ann__epochs": [50, 100],
}

total_combinations = (
    len(param_grid["ann__model__neurons"])
    * len(param_grid["ann__model__layers"])
    * len(param_grid["ann__epochs"])
)

print(f"Grid combinations: {total_combinations}")
print(
    "With 3-fold CV, this will train",
    total_combinations * 3,
    "ANN models."
)

Grid combinations: 12
With 3-fold CV, this will train 36 ANN models.


## 7. Run Grid Search

`GridSearchCV` will train every candidate configuration using 3-fold cross-validation on the training set.

I am using negative MAE as the scoring function because Scikit-learn expects a score where larger is better. Therefore, it represents the negative of the actual MAE during the search.

The final test set is still not used here.

In [8]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=1,
    verbose=1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'ann__epochs': [50, 100], 'ann__model__layers': [1, 2], 'ann__model__neurons': [16, 32, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to

## 8. Inspect the Best Configuration

The best configuration is the hyperparameter combination that produced the strongest average cross-validation score on the training data.

This does not mean it is guaranteed to be the best model on unseen data. That is why I still keep the final test set separate.

In [9]:
print("Best parameters:")
print(grid_search.best_params_)

print(
    f"Best CV MAE: {-grid_search.best_score_:,.2f}"
)

Best parameters:
{'ann__epochs': 100, 'ann__model__layers': 2, 'ann__model__neurons': 64}
Best CV MAE: 49,846.53


## 9. Review the Search Results

Looking at the full search table helps me understand whether the best configuration is clearly better than the alternatives or whether several architectures perform similarly.

This is often more useful than looking only at the single winning configuration.

In [11]:
results = pd.DataFrame(
    grid_search.cv_results_
)

results[
    [
        "param_ann__model__neurons",
        "param_ann__model__layers",
        "param_ann__epochs",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values(
    "rank_test_score"
)

,param_ann__model__neurons,param_ann__model__layers,param_ann__epochs,mean_test_score,std_test_score,rank_test_score
11,64,2,100,-49846.534120,291.130307,1
10,32,2,100,-49867.870353,319.536961,2
5,64,2,50,-49877.613846,284.950819,3
9,16,2,100,-49929.414609,297.439311,4
4,32,2,50,-49956.383593,315.889374,5
3,16,2,50,-50265.183095,359.043992,6
8,64,1,100,-75629.847170,1782.581768,7
7,32,1,100,-85535.962248,1845.398134,8
2,64,1,50,-91941.236211,1961.201273,9
6,16,1,100,-93006.426316,1910.610474,10


## 10. Evaluate the Tuned Model on the Unseen Test Set

The best configuration has now been selected using training-set cross-validation.

Only now will I evaluate it on the untouched test set.

These metrics provide the final comparison against the original regression experiment.

In [12]:
# The best estimator has already been refit on the full training set
# by GridSearchCV because refit=True is the default.
best_model = grid_search.best_estimator_

# Generate predictions on the unseen test set.
y_pred = best_model.predict(X_test)

# Calculate final regression metrics.
tuned_mae = mean_absolute_error(
    y_test,
    y_pred
)

tuned_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

tuned_r2 = r2_score(
    y_test,
    y_pred
)

print(f"Tuned ANN MAE  : {tuned_mae:,.2f}")
print(f"Tuned ANN RMSE : {tuned_rmse:,.2f}")
print(f"Tuned ANN R²   : {tuned_r2:.4f}")

Tuned ANN MAE  : 50,439.92
Tuned ANN RMSE : 58,350.17
Tuned ANN R²   : -0.0313


## 11. Compare Against a Simple Baseline

Hyperparameter tuning is only useful when the tuned model actually improves on a simple reference point.

Here I use the median salary from the training set as a baseline prediction for every test example.

In [13]:
# Predict the training-set median for every test observation.
baseline_predictions = np.full(
    len(y_test),
    y_train.median()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

comparison = pd.DataFrame({
    "Model": [
        "Median Baseline",
        "Tuned ANN"
    ],
    "MAE": [
        baseline_mae,
        tuned_mae
    ],
    "RMSE": [
        baseline_rmse,
        tuned_rmse
    ],
    "R2": [
        baseline_r2,
        tuned_r2
    ]
})

comparison

,Model,MAE,RMSE,R2
0,Median Baseline,49782.141120,57483.534944,-0.000940
1,Tuned ANN,50439.917423,58350.165679,-0.031348


## 12. What This Experiment Tells Me

Hyperparameter tuning answers a specific question:

> Can a different ANN architecture improve the model's performance on the same feature set?

If the tuned ANN remains close to the baseline, that is useful evidence that simply changing the network architecture is not solving the main limitation.

In that case, the next improvement should come from the **data and feature signal**, not just a larger neural network.

This is an important practical ML lesson: hyperparameter tuning optimizes the model configuration, but it cannot create predictive information that is not present in the input features.